<a href="https://colab.research.google.com/github/Pemoreira74/Clases_IA_2026/blob/main/IA_Clase2_practica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# 1. Cargar el conjunto de datos
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(f"Tamaño total del dataset: {X.shape} pacientes con {X.shape[1]} características.")

In [ ]:
print("Primeras 5 filas del dataset:")
print(X.head())

print("\nColumnas del dataset:")
print(X.columns)

print("\nInformación detallada del dataset:")
X.info()

In [ ]:
# 2. Partición de Datos (Estrategia Train / Dev / Test)
# Primero separamos el Train (80%) del resto (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Luego dividimos el 'resto' en Dev (10%) y Test (10%)
X_dev, X_test, y_dev, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Datos de Entrenamiento (Train): {X_train.shape}")
print(f"Datos de Desarrollo (Dev): {X_dev.shape}")
print(f"Datos de Prueba (Test): {X_test.shape}")

In [ ]:
# 3. Construcción del Modelo Base (Baseline)
# El dataset no está escalado y le cuesta converger
# verbose=2 imprimirá el progreso de la convergencia
modelo_base = LogisticRegression(C=1, max_iter=50, verbose=2)
modelo_base.fit(X_train, y_train)

In [ ]:
# 4. Evaluación en el conjunto de Desarrollo (Dev)
y_pred_dev = modelo_base.predict(X_dev)

# Cálculos de Métricas
matriz_conf = confusion_matrix(y_dev, y_pred_dev)
exactitud = accuracy_score(y_dev, y_pred_dev)
precision = precision_score(y_dev, y_pred_dev)
recall = recall_score(y_dev, y_pred_dev)
f1 = f1_score(y_dev, y_pred_dev) # Nuestra métrica de número único

print("\n--- RESULTADOS EN EL CONJUNTO DEV ---")
print("Matriz de Confusión:")
print(matriz_conf)
print(f"Exactitud (Accuracy): {exactitud:.4f}")
print(f"Precisión:            {precision:.4f}")
print(f"Exhaustividad (Recall):{recall:.4f}")
print(f"Puntaje F1 (F1-Score): {f1:.4f}  <-- Métrica de número único")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Crear un mapa de calor de la matriz de confusión
plt.figure(figsize=(6, 5))
sns.heatmap(matriz_conf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.xlabel('Predicción del Modelo')
plt.ylabel('Valores Reales')
plt.title('Matriz de Confusión')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import learning_curve
from sklearn.metrics import f1_score, make_scorer

# Definir el scorer para F1-score
f1_scorer = make_scorer(f1_score)

# Generar la curva de aprendizaje
train_sizes, train_scores, test_scores = learning_curve(
    modelo_base, X_train, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring=f1_scorer
)

# Calcular medias y desviaciones estándar
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

# Plotear la curva de aprendizaje
plt.figure(figsize=(10, 6))
plt.title("Curva de Aprendizaje (F1-Score) para Logistic Regression")
plt.xlabel("Tamaño del conjunto de entrenamiento")
plt.ylabel("F1-Score")
plt.grid()

plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, alpha=0.1,
                 color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std, alpha=0.1,
                 color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r",
         label="Puntuación de entrenamiento (F1)")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g",
         label="Puntuación de validación cruzada (F1)")

plt.legend(loc="best")
plt.show()

# Reflexión final:
¿Qué combinación no es aceptable en este contexto médico?

¿Por qué?

#Y si escalamos los datos?

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Creamos el "escalador" que convertirá todos los datos a la misma escala (media 0, varianza 1)
scaler = StandardScaler()

# 2. Ajustamos y transformamos los datos de entrenamiento
X_train_escalado = scaler.fit_transform(X_train)

# 3. Transformamos los datos de desarrollo (¡Ojo! Solo transformamos, no ajustamos)
X_dev_escalado = scaler.transform(X_dev)

# 4. Volvemos a entrenar nuestro modelo "débil" de 50 iteraciones, pero con los datos escalados
modelo_base.fit(X_train_escalado, y_train)

# 5. Volvemos a evaluar
predicciones_nuevas = modelo_base.predict(X_dev_escalado)
print("Nueva Exactitud con datos escalados:", accuracy_score(y_dev, predicciones_nuevas))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Calculamos la matriz matemática comparando los valores reales vs las nuevas predicciones
matriz_corregida = confusion_matrix(y_dev, predicciones_nuevas)

# 2. Configuramos el tamaño del gráfico para que se vea bien en Jupyter
plt.figure(figsize=(6, 4))

# 3. Dibujamos el mapa de calor (heatmap)
# annot=True pone los números dentro de las cajas
# fmt='d' asegura que los números sean enteros (sin decimales)
# cmap='Blues' le da un tono azul profesional, pero puedes usar 'viridis' o 'Reds'
sns.heatmap(matriz_corregida, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benigno', 'Maligno'],
            yticklabels=['Benigno', 'Maligno'])

# 4. Añadimos título y etiquetas a los ejes para que sea fácil de interpretar
plt.title('Matriz de Confusión - Modelo con Datos Escalados')
plt.xlabel('Predicción del Modelo')
plt.ylabel('Valor Real (Ground Truth)')

# 5. Renderizamos el gráfico
plt.show()

In [ ]:
# Definir el scorer para F1-score
f1_scorer = make_scorer(f1_score)

# Generar la curva de aprendizaje
# Usamos los datos de entrenamiento escalados, ya que así se entrenó el modelo final
train_sizes, train_scores, test_scores = learning_curve(
    modelo_base, X_train_escalado, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring=f1_scorer
)

# Calcular medias y desviaciones estándar
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

# Plotear la curva de aprendizaje
plt.figure(figsize=(10, 6))
plt.title("Curva de Aprendizaje (F1-Score) para Logistic Regression")
plt.xlabel("Tamaño del conjunto de entrenamiento")
plt.ylabel("F1-Score")
plt.grid()

plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, alpha=0.1,
                 color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std, alpha=0.1,
                 color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r",
         label="Puntuación de entrenamiento (F1)")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g",
         label="Puntuación de validación cruzada (F1)")

plt.legend(loc="best")
plt.show()

#Vamos a abrir el capot del modelo

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Obtenemos el intercepto (el sesgo base del modelo)
sesgo = modelo_base.intercept_[0]
print(f"El punto de partida de la ecuación (Intercepto) es: {sesgo:.4f}\n")

# 2. Obtenemos los nombres de las 30 columnas (características) del DataFrame original
nombres_caracteristicas = X.columns

# 3. Extraemos los 30 coeficientes calculados por el modelo
coeficientes = modelo_base.coef_[0]

# 4. Creamos una tabla (DataFrame) para emparejar cada característica con su peso matemático
importancia_variables = pd.DataFrame({
    'Característica': nombres_caracteristicas,
    'Peso (Coeficiente)': coeficientes
})

# 5. Calculamos el "Impacto Absoluto" (sin importar si es positivo o negativo, queremos ver su fuerza)
importancia_variables['Impacto Absoluto'] = importancia_variables['Peso (Coeficiente)'].abs()

# 6. Ordenamos la tabla de mayor a menor impacto para ver las más importantes arriba
importancia_variables = importancia_variables.sort_values(by='Impacto Absoluto', ascending=False)

# --- VISUALIZACIÓN ---
plt.figure(figsize=(10, 6))

# Graficamos el Top 10 de características
sns.barplot(
    x='Peso (Coeficiente)',
    y='Característica',
    data=importancia_variables.head(10),
    palette='vlag' # Colores que diferencian valores negativos de positivos
)

plt.title('Top 10 Características que definen la Frontera de Decisión', fontsize=14)
plt.xlabel('Peso en la Ecuación (Fuerza y Dirección)', fontsize=12)
plt.ylabel('Característica del Tumor', fontsize=12)
plt.axvline(0, color='black', linewidth=1) # Línea en el cero para separar positivo de negativo
plt.show()

# Imprimimos también los valores exactos
print(importancia_variables.head(10)[['Característica', 'Peso (Coeficiente)']])

Un peso negativo en la ecuación significa que, a medida que el valor de esa característica aumenta, el resultado matemático se hace más pequeño, empujando la predicción con mucha fuerza hacia el 0 (Maligno)

# Tarea
Calcule las métricas de rendimiento (puntuación F1, exactitud, precisión, recuperación) del `modelo_base` en el conjunto de datos de entrenamiento (`X_train`, `y_train`).

## Calcular Métricas en el Conjunto de Entrenamiento

### Subtarea:

Calcular las métricas de rendimiento (F1-score, exactitud, precisión, exhaustividad) del `modelo_base` en el conjunto de datos de entrenamiento (`X_train`, `y_train`).


**Razonamiento**:
La subtarea requiere calcular e imprimir métricas de rendimiento (puntuación F1, exactitud, precisión y recuperación) para el `modelo_base` en el conjunto de datos de entrenamiento. Esto implica realizar predicciones, calcular la matriz de confusión y, posteriormente, calcular cada métrica. Todas estas acciones se pueden realizar en un solo bloque de código.



## Calcular Métricas en el Conjunto de Prueba

### Subtarea:
Calcular las métricas de rendimiento (F1-score, exactitud, precisión, exhaustividad) del `modelo_base` en el conjunto de datos de prueba (`X_test`, `y_test`).


**Razonamiento**:
La subtarea requiere calcular e imprimir métricas de rendimiento (puntuación F1, exactitud, precisión y recuperación) para el `modelo_base` en el conjunto de datos de prueba. Esto implica realizar predicciones, calcular la matriz de confusión y, posteriormente, calcular cada métrica. Todas estas acciones se pueden realizar en un solo bloque de código.



## Comparar y Visualizar Métricas

### Subtarea:
Consolidar las métricas de F1-score de los conjuntos de entrenamiento, desarrollo y prueba, y crear un gráfico de barras para visualizar y comparar estos resultados. Asegúrate de incluir etiquetas claras para cada conjunto (Train, Dev, Test).


## Tarea final

### Subtarea:
Proporcione un resumen de la comparación del rendimiento del modelo en los conjuntos de entrenamiento, desarrollo y prueba.